# Q9.
```{admonition}
:class: note
In this exercise, we will predict the number of applications received using the other variables in the `College` data set.

In [76]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.model_selection import KFold, train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
college = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/College.csv',index_col=0)
college['Private'] = college['Private'] == 'Yes'

In [35]:
college.sample(4)

,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
Oberlin College,True,4778,2767,678,50,89,2587,120,19670,5820,575,1119,77,96,10.1,47,16593,83
Hood College,True,699,565,176,36,64,710,399,13960,6040,450,690,82,88,14.4,34,12434,72
Barat College,True,261,192,111,15,36,453,266,9690,4300,500,500,57,77,9.7,35,9337,71
Missouri Southern State College,False,1576,1326,913,13,50,3689,2200,3840,2852,200,400,52,54,20.3,9,4172,100


## (a)
```{admonition}
:class: note
Split the data set into a training set and a test set.

In [57]:
X_train, X_test, Y_train, Y_test = train_test_split(college.drop(columns=['Apps']),college[['Apps']],random_state=1728)

## (b)
```{admonition}
:class: note
Fit a linear model using least squares on the training set, and report the test error obtained.

In [92]:
lr = LinearRegression()
lr.fit(X_train,Y_train)
lr_mse = mean_squared_error(Y_test,lr.predict(X_test))
print(f'{lr_mse}')

1357522.7450599347


## (c)
```{admonition}
:class: note
Fit a ridge regression model on the training set, with $\lambda$ chosen by cross-validation. Report the test error obtained.

In [102]:
pipe1 = Pipeline(
    [
        ('scale',StandardScaler()),('ridge',RidgeCV(alphas=np.logspace(-2,3,200),cv=10))
    ]
)
pipe1.fit(X_train,Y_train)
print(pipe1.named_steps['ridge'].alpha_)
ridge_mse = mean_squared_error(Y_test,pipe1.predict(X_test))
print(f'{ridge_mse}')

6.150985788580504
1302149.4027441759


## (d)
```{admonition}
:class: note
Fit a lasso model on the training set, with $\lambda$ chosen by cross-validation. Report the test error obtained, along with the number of non-zero coefficient estimates.

In [103]:
pipe2 = Pipeline(
    [
        ('scale',StandardScaler()),('lasso',LassoCV(alphas=np.logspace(-2,3,200),cv=10))
    ]
)
pipe2.fit(X_train,Y_train.values.ravel())
print(pipe2.named_steps['lasso'].alpha_)
lasso_mse = mean_squared_error(Y_test,pipe2.predict(X_test))
print(f'{lasso_mse}')

0.01
1357513.8780917858


## (e)
```{admonition}
:class: note
Fit a PCR model on the training set, with M chosen by cross-validation. Report the test error obtained, along with the value of $M$ selected by cross-validation.

In [82]:
pipe3 = Pipeline(
    [
        ('scale',StandardScaler()),('pca',PCA()),('regression',LinearRegression())
    ]
)
gridsearch = GridSearchCV(pipe3,{'pca__n_components':range(1,X_train.shape[1]+1)},scoring='neg_mean_squared_error')
gridsearch.fit(X_train,Y_train)
print(gridsearch.best_params_)

{'pca__n_components': 17}
1262512.767813548


In [88]:
pipe3.named_steps['pca'].n_components = gridsearch.best_params_['pca__n_components']
pipe3.fit(X_train,Y_train)
pcr_mse= mean_squared_error(Y_test,pipe3.predict(X_test))
print(f'{pcr_mse}')

1357522.7450599254


## (f)
```{admonition}
:class: note
Fit a PLS model on the training set, with M chosen by cross-validation. Report the test error obtained, along with the value of $M$ selected by cross-validation.

In [85]:
pls = PLSRegression(scale=True)
gridsearch2 = GridSearchCV(pls,{'n_components':range(1,X_train.shape[1]+1)},scoring='neg_mean_squared_error')
gridsearch2.fit(X_train,Y_train)
print(gridsearch2.best_params_)

{'n_components': 14}


In [87]:
pls.n_components = gridsearch2.best_params_['n_components']
pls.fit(X_train,Y_train)
pls_mse = mean_squared_error(Y_test,pls.predict(X_test))
print(f'{pls_mse}')

1358160.3958477273


## (g)
```{admonition}
:class: note
Comment on the results obtained. How accurately can we predict the number of college applications received? Is there much difference among the test errors resulting from these five approaches?

In [104]:
mse_list = [lr_mse,ridge_mse,lasso_mse,pcr_mse,pls_mse]

for mse in mse_list:
    print(np.sqrt(mse))

1165.1277805716995
1141.1176112672067
1165.1239754171168
1165.1277805716957
1165.4013882983525


In [105]:
from sklearn.metrics import r2_score
print(r2_score(Y_test,lr.predict(X_test)))
print(r2_score(Y_test,pipe1.predict(X_test)))
print(r2_score(Y_test,pipe2.predict(X_test)))
print(r2_score(Y_test,pipe3.predict(X_test)))
print(r2_score(Y_test,pls.predict(X_test)))

0.8988492577049217
0.9029751957041051
0.8988499183939698
0.8988492577049224
0.8988017455356082


Each model is comprable to the others, but ridge performed slightly better.